# Dependências

In [ ]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
!pip install gcloud
!gcloud auth application-default login

# Import necessary Python libraries
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.4/454.4 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=602927 sha256=d60b0768e502d12a70722b8ce426336793a0a6b79c92cda68bf5b8a09f16d4ef
  Stored in directory: /root/.cache/pip/wheels/2a/62/75/3d74209bfebb8805823ae74afa28653aa1ea76d8b5a9d741ff
Successfully built gcloud
Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=t50QwLCLU0znQ5fRcSULVhzPy07A9d&prompt=consent&token_usage=remote&access_type=offline&code_chal

# Tratamento

Esta versão parte de uma tabela cuja estrutura é:

`ano | cor | genero | freq | freq_se | freq_cv | prop | prop_se | prop_cv | tipo_representacao`

Diferente do fluxo antigo (que unia duas planilhas com colunas `sum`, `sexo`, `cor`, `cv`),
aqui a `freq` já é a quantidade de vínculos/servidores e `prop` já vem calculada.
O tratamento se resume a: selecionar as colunas úteis, renomear para o padrão do dataset
(`cor_raca`, `quantidade_vinculos`), arredondar a frequência para inteiro e ajustar a
proporção para porcentagem (0–100), mantendo o mesmo esquema final de saída.

In [ ]:
# Leitura da fonte com a nova estrutura de colunas
# (ajuste o nome do arquivo e da aba conforme necessário)
df = pd.read_excel('/content/indicador_pnad_lideranca_genero_cor.xlsx')
df = df[df['tipo_representacao'] == 'Servidores em cargos de liderança']

## Seleção e padronização das colunas

In [ ]:
# Mantém apenas as colunas necessárias para o dataset final
df = df[['ano', 'cor', 'genero', 'freq', 'prop']].copy()

# Renomeia para o padrão do dataset de destino
df = df.rename(columns={
    'cor': 'cor_raca',
    'freq': 'quantidade_vinculos'
})

# freq é a contagem de vínculos/servidores -> arredonda para inteiro
df['quantidade_vinculos'] = df['quantidade_vinculos'].round().astype('Int64')

# prop na fonte vem como fração (0–1). O dataset final usa porcentagem (0–100),
# assim como no fluxo antigo (prop_genero_cor_ano em %).
df['prop'] = (df['prop'] * 100)

# Ordena as colunas no layout final
df = df[['ano', 'cor_raca', 'genero', 'quantidade_vinculos', 'prop']]
df

,ano,cor_raca,genero,quantidade_vinculos,prop
6,2013,Branca,Homem,169019,37.857700
7,2013,Branca,Mulher,112671,25.236475
8,2013,Negra,Homem,111089,24.882138
9,2013,Negra,Mulher,51979,11.642475
10,2013,Outra,Homem,31,0.006836
...,...,...,...,...,...
163,2026,Branca,Mulher,83030,23.871749
164,2026,Negra,Homem,90233,25.942775
165,2026,Negra,Mulher,58664,16.866359
166,2026,Outra,Homem,1961,0.563836


In [ ]:
df = df[df['ano'].isin([2024, 2025, 2026])]

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18 entries, 138 to 167
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ano                  18 non-null     int64  
 1   cor_raca             18 non-null     object 
 2   genero               18 non-null     object 
 3   quantidade_vinculos  18 non-null     Int64  
 4   prop                 18 non-null     float64
dtypes: Int64(1), float64(1), int64(1), object(2)
memory usage: 882.0+ bytes


### Verificações rápidas (opcional)

Confere se as proporções fecham em ~100% por ano e inspeciona os tipos.

In [ ]:
# Soma de prop por ano (deve ficar próxima de 100)
df.groupby(['ano', 'genero'])['prop'].sum()

ano   genero
2024  Homem     60.187723
      Mulher    39.812277
2025  Homem     60.026578
      Mulher    39.973422
2026  Homem     58.837366
      Mulher    41.162634
Name: prop, dtype: float64

In [ ]:
df.info()

### (Opcional) Consolidar por ano/gênero/cor

Se a fonte tiver mais de uma linha por combinação `ano/genero/cor_raca`,
agregue e recalcule a proporção — análogo ao groupby do notebook original.
Se a fonte já vem com uma linha por combinação, **pule esta célula**.

In [ ]:
# df = (
#     df.groupby(['ano', 'genero', 'cor_raca'], as_index=False)
#       .agg({'quantidade_vinculos': 'sum'})
# )
#
# # total por ano
# df['total_ano'] = df.groupby('ano')['quantidade_vinculos'].transform('sum')
#
# # proporção recalculada (%)
# df['prop'] = (df['quantidade_vinculos'] / df['total_ano']) * 100
#
# df = df[['ano', 'cor_raca', 'genero', 'quantidade_vinculos', 'prop']]
# df.head()

# Upload

In [ ]:
client = bigquery.Client(project='repositoriodedadosgpsp')

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [ ]:
schema = [bigquery.SchemaField('ano', 'INTEGER', description= 'Ano de referência da observação'),
          bigquery.SchemaField('cor_raca', 'STRING', description= 'Raça/cor autodeclarado ou não'),
          bigquery.SchemaField('genero', 'STRING', description= 'Gênero autodeclarado ou não'),
          bigquery.SchemaField('quantidade_vinculos', 'INTEGER', description= 'Número total de vinculos observados'),
          bigquery.SchemaField('prop', 'FLOAT', description= 'Proporção de vínculos em relação ao total naquele ano'),
          ]

dataset_ref = client.dataset('cargos_lideranca')

table_ref = dataset_ref.table('PNAD_vinculos_lideranca_genero_cor_v1')
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result()

LoadJob<project=repositoriodedadosgpsp, location=US, id=cc6c909f-4e47-4dfd-8b6c-67647a97cf12>